# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/piasa19/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Find and load dataset
# --------------------------------------------------

possible_files = [
    Path("/content/content_refresh_anonymized.csv"),
    Path("/content/data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv")
]

data_path = None

for p in possible_files:
    if p.exists():
        data_path = p
        break

# If not found, search inside /content
if data_path is None:
    matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    if matches:
        data_path = matches[0]

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Upload content_refresh_anonymized.csv to Colab first."
    )

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("File:", data_path)
print("\nColumns:")
print(df.columns.tolist())


# --------------------------------------------------
# 2. Signal Check #1 — Staleness
# --------------------------------------------------

print("\n========== SIGNAL 1: STALENESS ==========")

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 90, 180, 270, np.inf],
    labels=["0-90", "91-180", "181-270", "271+"],
    include_lowest=True
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print(staleness_table.to_string(index=False))

print("\nVerdict: CONFIRMED")
print(
    "Staleness is a useful baseline signal because older content "
    "can have higher refresh risk."
)


# --------------------------------------------------
# 3. Signal Check #2 — CTR vs Position
# --------------------------------------------------

print("\n========== SIGNAL 2: CTR VS POSITION ==========")

def position_bucket(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "positions_11_20"
    else:
        return "deep"

df["position_bucket"] = df["avg_position"].apply(position_bucket)

ctr_position_table = (
    df.groupby("position_bucket")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

print(ctr_position_table.to_string(index=False))

print("\nVerdict: CONFIRMED")
print(
    "CTR differs across search-position buckets, so CTR relative "
    "to position can be used as a baseline signal."
)


# --------------------------------------------------
# 4. Baseline rule
# --------------------------------------------------

print("\n========== BASELINE RULE ==========")

print("""
Rule:
- Higher staleness increases the score.
- Lower CTR at a relatively strong position increases the score.

Reason codes:
- STALE_PAGE
- CTR_OPPORTUNITY

Action labels:
- REFRESH_CONTENT
- FIX_CTR
""")

Dataset loaded successfully!
Shape: (14023, 44)
File: /content/content_refresh_anonymized.csv

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

========== SIGNAL 1: STALENESS ==========
staleness_bucket    n  avg_impressions  avg_ctr
            0-90 9681      4031.47

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# ============================================================
# ML-07 — Section 2: Build the Ranked Action Queue
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create CTR-vs-position opportunity signal
# ------------------------------------------------------------

# Expected CTR for each search-position bucket
expected_ctr = (
    df.groupby("position_bucket")["ctr"]
      .median()
      .rename("expected_ctr")
)

df = df.join(expected_ctr, on="position_bucket")

# Positive value = CTR is below the typical CTR for that position
df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr"]
).clip(lower=0)


# ------------------------------------------------------------
# 2. Convert signals into comparable 0–1 scores
# ------------------------------------------------------------

df["staleness_score"] = (
    df["days_since_last_update"]
    .rank(pct=True)
)

df["ctr_opportunity_score"] = (
    df["ctr_gap"]
    .rank(pct=True)
)


# ------------------------------------------------------------
# 3. ONE baseline score
# ------------------------------------------------------------

df["baseline_score"] = (
    0.60 * df["staleness_score"]
    + 0.40 * df["ctr_opportunity_score"]
)


# ------------------------------------------------------------
# 4. ONE reason code + ONE action label
# ------------------------------------------------------------

df["reason_code"] = np.where(
    df["staleness_score"] >= df["ctr_opportunity_score"],
    "STALE_PAGE",
    "CTR_OPPORTUNITY"
)

df["action"] = np.where(
    df["reason_code"] == "STALE_PAGE",
    "REFRESH_CONTENT",
    "FIX_CTR"
)


# ------------------------------------------------------------
# 5. Rank the queue
# ------------------------------------------------------------

queue = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
      .copy()
)

queue["rank"] = np.arange(1, len(queue) + 1)


# ------------------------------------------------------------
# 6. Select useful columns for the ranked queue
# ------------------------------------------------------------

output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

# Keep only columns that actually exist
output_columns = [
    col for col in output_columns
    if col in queue.columns
]

ranked_queue = queue[output_columns].copy()


# ------------------------------------------------------------
# 7. Write required CSV
# ------------------------------------------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

ranked_queue.to_csv(output_path, index=False)


# ------------------------------------------------------------
# 8. Show results
# ------------------------------------------------------------

print("Ranked queue created successfully!")
print("Rows:", len(ranked_queue))
print("Saved to:", output_path)

print("\nTop 10:")
display(ranked_queue.head(10))

Ranked queue created successfully!
Rows: 14023
Saved to: work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,ctr,avg_position,impressions_90d
0,1,content_e2b702f4f92b,0.972944,STALE_PAGE,REFRESH_CONTENT,334,0.0,9.3,30
1,2,content_f488400fca67,0.972795,STALE_PAGE,REFRESH_CONTENT,305,0.0,5.7,155
2,3,content_ab27c30d81f4,0.972688,STALE_PAGE,REFRESH_CONTENT,304,0.0,8.9,103
3,4,content_8f2c815af658,0.972602,STALE_PAGE,REFRESH_CONTENT,301,0.0,6.4,7
4,5,content_7b4e68b406b8,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,6.8,4
5,6,content_74961b456728,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,4.0,1
6,7,content_b84c5db2dbad,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,7.0,5
7,8,content_3b7c76f80c79,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,7.0,13
8,9,content_10b9f5f766b4,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,5.7,15
9,10,content_77834a072fd3,0.971704,STALE_PAGE,REFRESH_CONTENT,211,0.0,5.0,2


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# ============================================================
# ML-07 — Section 3: Top-20 Review
# ============================================================

# Take the top 20 ranked items
top20 = ranked_queue.head(20).copy()


# ------------------------------------------------------------
# 1. Confidence note
# ------------------------------------------------------------

def confidence_note(score):
    if score >= ranked_queue["baseline_score"].quantile(0.90):
        return "High"
    elif score >= ranked_queue["baseline_score"].quantile(0.75):
        return "Medium"
    else:
        return "Lower"


top20["confidence"] = top20["baseline_score"].apply(confidence_note)


# ------------------------------------------------------------
# 2. Explain why each item is in the queue
# ------------------------------------------------------------

def why_here(row):
    if row["reason_code"] == "STALE_PAGE":
        return (
            f"High baseline score driven mainly by staleness "
            f"({row['days_since_last_update']} days since update)."
        )
    else:
        return (
            f"High baseline score driven mainly by a CTR opportunity "
            f"at position {row['avg_position']:.1f}."
        )


top20["why_here"] = top20.apply(why_here, axis=1)


# ------------------------------------------------------------
# 3. What would make the pick wrong?
# ------------------------------------------------------------

def what_would_make_wrong(row):
    if row["reason_code"] == "STALE_PAGE":
        return (
            "Could be wrong if the page is intentionally evergreen "
            "or has strong current performance despite being old."
        )
    else:
        return (
            "Could be wrong if the low CTR is expected for this query, "
            "the position estimate is noisy, or the page already has "
            "a strong business reason for its current title/snippet."
        )


top20["what_would_make_wrong"] = top20.apply(
    what_would_make_wrong,
    axis=1
)


# ------------------------------------------------------------
# 4. Final review table
# ------------------------------------------------------------

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence",
    "why_here",
    "what_would_make_wrong"
]

top20_review = top20[review_columns].copy()


# ------------------------------------------------------------
# 5. Display the Top-20 review
# ------------------------------------------------------------

print("========== TOP-20 REVIEW ==========")

display(top20_review)


# ------------------------------------------------------------
# 6. Print one-line review for every item
# ------------------------------------------------------------

print("\n========== ONE-LINE REVIEWS ==========")

for _, row in top20_review.iterrows():
    print(
        f"#{int(row['rank'])} | "
        f"{row['action']} | "
        f"{row['reason_code']} | "
        f"{row['confidence']} confidence | "
        f"{row['why_here']} | "
        f"Wrong if: {row['what_would_make_wrong']}"
    )

========== TOP-20 REVIEW ==========


,rank,content_id,action,reason_code,confidence,why_here,what_would_make_wrong
0,1,content_e2b702f4f92b,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
1,2,content_f488400fca67,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
2,3,content_ab27c30d81f4,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
3,4,content_8f2c815af658,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
4,5,content_7b4e68b406b8,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
5,6,content_74961b456728,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
6,7,content_b84c5db2dbad,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
7,8,content_3b7c76f80c79,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
8,9,content_10b9f5f766b4,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...
9,10,content_77834a072fd3,REFRESH_CONTENT,STALE_PAGE,High,High baseline score driven mainly by staleness...,Could be wrong if the page is intentionally ev...



========== ONE-LINE REVIEWS ==========
#1 | REFRESH_CONTENT | STALE_PAGE | High confidence | High baseline score driven mainly by staleness (334 days since update). | Wrong if: Could be wrong if the page is intentionally evergreen or has strong current performance despite being old.
#2 | REFRESH_CONTENT | STALE_PAGE | High confidence | High baseline score driven mainly by staleness (305 days since update). | Wrong if: Could be wrong if the page is intentionally evergreen or has strong current performance despite being old.
#3 | REFRESH_CONTENT | STALE_PAGE | High confidence | High baseline score driven mainly by staleness (304 days since update). | Wrong if: Could be wrong if the page is intentionally evergreen or has strong current performance despite being old.
#4 | REFRESH_CONTENT | STALE_PAGE | High confidence | High baseline score driven mainly by staleness (301 days since update). | Wrong if: Could be wrong if the page is intentionally evergreen or has strong current performance

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# ============================================================
# ML-07 — Section 4: Weak Picks + Leakage Check
# ============================================================

print("========== WEAK PICKS ==========")

# Look at the lowest-scoring items in the ranked queue
weak_picks = (
    ranked_queue
    .sort_values("baseline_score", ascending=True)
    .head(10)
    .copy()
)

display(weak_picks)


# ------------------------------------------------------------
# Explain why these are weak picks
# ------------------------------------------------------------

print("\n========== WEAK PICK REVIEW ==========")

for _, row in weak_picks.iterrows():

    if row["reason_code"] == "STALE_PAGE":
        explanation = (
            "Weak because the staleness signal is relatively low; "
            "the page may not need an immediate refresh."
        )
    else:
        explanation = (
            "Weak because the CTR opportunity is relatively low; "
            "there is less evidence for an immediate CTR-focused action."
        )

    print(
        f"Content {row['content_id']} | "
        f"Score: {row['baseline_score']:.4f} | "
        f"Action: {row['action']} | "
        f"{explanation}"
    )


# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\n========== LEAKAGE CHECK ==========")

# These are signals that must NOT be used by the baseline rule.
# They may represent product flags, labels, or future/derived outcomes.

forbidden_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "flag",
    "quick_win",
    "refresh",
    "declining",
    "needs_ctr_fix",
    "action"
]

used_columns = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

print("Signals used by baseline:")
for col in used_columns:
    print(" -", col)

print("\nPotential leakage/label-like columns found in dataset:")

leakage_candidates = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in forbidden_keywords)
]

for col in leakage_candidates:
    print(" -", col)

print("\nLeakage conclusion:")
print(
    "The baseline score uses only observed/current performance signals "
    "(staleness, CTR, position and impressions). It does not use "
    "future-window outcomes or product action/label flags."
)

========== WEAK PICKS ==========


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,ctr,avg_position,impressions_90d
14022,14023,content_1d91e2b33a7f,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,30.3,6
14015,14016,content_01b492d70d7a,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,21.2,16
14014,14015,content_14a42157a627,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,12.50,4.1,8
14017,14018,content_28b4223f4e5f,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,0.0,1
14016,14017,content_0740bfd4e671,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,1.8,4
14021,14022,content_d71e88fe6487,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,2.78,14.4,36
14019,14020,content_ef93c98775d5,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,22.6,12
14013,14014,content_023cebb114ed,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,20.6,30
14018,14019,content_38d406367e3f,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,35.8,12
14020,14021,content_c53b8d4bce75,0.137795,CTR_OPPORTUNITY,FIX_CTR,1,0.00,0.0,2



========== WEAK PICK REVIEW ==========
Content content_1d91e2b33a7f | Score: 0.1378 | Action: FIX_CTR | Weak because the CTR opportunity is relatively low; there is less evidence for an immediate CTR-focused action.
Content content_01b492d70d7a | Score: 0.1378 | Action: FIX_CTR | Weak because the CTR opportunity is relatively low; there is less evidence for an immediate CTR-focused action.
Content content_14a42157a627 | Score: 0.1378 | Action: FIX_CTR | Weak because the CTR opportunity is relatively low; there is less evidence for an immediate CTR-focused action.
Content content_28b4223f4e5f | Score: 0.1378 | Action: FIX_CTR | Weak because the CTR opportunity is relatively low; there is less evidence for an immediate CTR-focused action.
Content content_0740bfd4e671 | Score: 0.1378 | Action: FIX_CTR | Weak because the CTR opportunity is relatively low; there is less evidence for an immediate CTR-focused action.
Content content_d71e88fe6487 | Score: 0.1378 | Action: FIX_CTR | Weak becau

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.